# E-commerce Medallion Pipeline — Full Run (Serverless)

Runs **Bronze → Silver → Gold** from your Databricks repo clone.

**Before you start:** Repos → **Pull** latest `main`.

**Serverless rules:**
- CSVs go in `{REPO_ROOT}/data` — not FileStore, not `/tmp`
- Run cells **1 → 5** in order (or **Run all**)

See also: `notebooks/databricks_serverless_pipeline.ipynb` (same flow, minimal UI).

In [ ]:
import os
import sys
from pathlib import Path

# Legacy widget kept for old cells / job parameters (value is optional)
dbutils.widgets.text("source_base_path", "", "Leave empty = {REPO_ROOT}/data")

notebook_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)

repo_rel = os.path.dirname(os.path.dirname(notebook_path))
REPO_ROOT = repo_rel if repo_rel.startswith("/Workspace") else f"/Workspace{repo_rel}"
SRC_ROOT = os.path.join(REPO_ROOT, "src")
DATA_DIR = Path(REPO_ROOT) / "data"

if not os.path.isdir(SRC_ROOT):
    raise FileNotFoundError(f"src not found at {SRC_ROOT} — open notebook from Git repo")

if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

os.environ["PIPELINE_REPO_ROOT"] = REPO_ROOT
import config.pipeline_config  # noqa: F401
from config.databricks_runtime import ensure_notebook_modules

ensure_notebook_modules()

from gold import create_gold_tables as _gold_tables_module

_recon_version = getattr(_gold_tables_module, "RECONCILIATION_LOGIC_VERSION", None)
if _recon_version is None:
    raise RuntimeError(
        "Stale pipeline code detected. Open the repo folder → Git → Pull latest main, "
        "then run dbutils.library.restartPython() and Run All again."
    )
print(f"pipeline_reconciliation_logic={_recon_version}")

DATA_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_BASE_PATH_WIDGET = dbutils.widgets.get("source_base_path").strip()
if SOURCE_BASE_PATH_WIDGET and "/FileStore" not in SOURCE_BASE_PATH_WIDGET and not SOURCE_BASE_PATH_WIDGET.startswith(("/tmp", "file:/tmp")):
    SOURCE_BASE_PATH = SOURCE_BASE_PATH_WIDGET
else:
    from config.databricks_runtime import to_spark_readable_path

    SOURCE_BASE_PATH = to_spark_readable_path(str(DATA_DIR.resolve()))

dbutils.widgets.text("schema_name", "ecommerce", "Schema")
dbutils.widgets.dropdown("generate_sample_data", "true", ["true", "false"], "Generate CSVs")
dbutils.widgets.text("catalog", "", "Catalog (optional)")
dbutils.widgets.text("run_id", "", "Run id (optional)")

SCHEMA_NAME = dbutils.widgets.get("schema_name").strip() or "ecommerce"
GENERATE_SAMPLE_DATA = dbutils.widgets.get("generate_sample_data") == "true"
CATALOG = dbutils.widgets.get("catalog").strip() or None
RUN_ID = dbutils.widgets.get("run_id").strip() or None

print(f"REPO_ROOT={REPO_ROOT}")
print(f"SOURCE_BASE_PATH={SOURCE_BASE_PATH}")
print(f"schema={SCHEMA_NAME} generate_sample_data={GENERATE_SAMPLE_DATA}")

In [ ]:
if GENERATE_SAMPLE_DATA:
    from data_generation.generate_sample_data import write_sample_datasets, DEFAULT_SEED

    print(f"Generating seed={DEFAULT_SEED} CSVs to {DATA_DIR} ...")
    write_sample_datasets(DATA_DIR, seed=DEFAULT_SEED)
else:
    missing = [n for n in ("customers.csv", "products.csv", "orders.csv") if not (DATA_DIR / n).is_file()]
    if missing:
        raise FileNotFoundError(f"Missing {missing} in {DATA_DIR}. Set generate_sample_data=true.")
    print(f"Using existing CSVs in {DATA_DIR}")

In [ ]:
import logging
import sys

from config.databricks_runtime import ensure_notebook_modules

ensure_notebook_modules()

import run_pipeline as _run_pipeline_module

sys.modules.setdefault("run_pipeline", _run_pipeline_module)

from config.pipeline_config import load_config
from run_pipeline import configure_logging, run_pipeline

configure_logging("INFO")

config = load_config(
    source_base_path=SOURCE_BASE_PATH,
    catalog=CATALOG,
    schema_name=SCHEMA_NAME,
    run_id=RUN_ID,
)

summary = run_pipeline(
    config,
    spark=spark,
    repo_root=REPO_ROOT,
    generate_sample_data_flag=False,
    validate_sample_data=True,
    strict_sample_row_counts=False,
)

print("\n=== Pipeline summary ===")
print(f"source_base_path: {summary.source_base_path}")
print(f"run_id: {summary.run_id}")
print(f"elapsed_seconds: {summary.elapsed_seconds:.1f}")
print(f"bronze: {summary.bronze_row_counts}")
print(f"silver: {summary.silver_row_counts}")
print(f"gold: {summary.gold_row_counts}")

In [ ]:
spark.sql(f"USE {SCHEMA_NAME}")

display(spark.sql("""
    SELECT 'bronze_orders' AS table_name, COUNT(*) AS row_count FROM bronze_orders
    UNION ALL SELECT 'gold_sales_by_product', COUNT(*) FROM gold_sales_by_product
    UNION ALL SELECT 'gold_customer_segmentation', COUNT(*) FROM gold_customer_segmentation
"""))